In [ ]:
# conda activate chronocell

import os
import sys
import time
import pandas as pd

os.chdir("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint")

# sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint/code")
# sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/simulations/code")

# import Chronocell
from reconstruct_RNA_history import *
# from protein_from_RNA import *

In [3]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.5_208_genes_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [4]:
Y = traj.X
Q = traj.Q[:, 0, :] 
tau = traj.tau # State transition times (global)
t = traj.t
theta = traj.theta
topo = traj.topo
state_grid = np.searchsorted(tau, t, side="left") - 1
state_grid[0] = 0 

# theta_ = theta.copy()
# a0 = theta_[:, 0] # Starting RNA abundance 
# a = theta_[:, 1:len(topo.flatten())] 
# beta = theta_[:, -2] # Splicing rate
# alpha = a * beta[:, None] # These values are divided by splicing rate; removing this factor now
# gamma = theta_[:, -1] # Degradation rate

## Subset to genes with protein measurements

### How many counts are in cells for genes of interest?

In [5]:
genes = pd.read_csv("eLNPs_var>1.5_208_genes.csv")
shared_genes = pd.read_csv("RNA_vs_ADT_corr_meanExpr.csv")

In [6]:
gene_idx = genes['Gene_symbol'].isin(shared_genes['Gene']).tolist()

In [7]:
shared_genes_mean_expr_U = Y[:, gene_idx, 0].max(axis=0)
shared_genes_mean_expr_S = Y[:, gene_idx, 1].max(axis=0)

In [8]:
np.sort(shared_genes_mean_expr_U)

array([  1.,   1.,   1.,   1.,   2.,   2.,   3.,   3.,   3.,   3.,   3.,
         3.,   3.,   4.,   4.,   4.,   4.,   5.,   6.,   6.,   6.,   6.,
         6.,   7.,   7.,   7.,   9.,  10.,  11.,  11.,  12.,  14.,  14.,
        16.,  16.,  16.,  19.,  19.,  22.,  22.,  24.,  25.,  27.,  27.,
        38.,  42.,  46.,  54.,  56.,  60., 115.])

In [9]:
np.sort(shared_genes_mean_expr_S)

array([  3.,   4.,   4.,   4.,   5.,   5.,   5.,   5.,   6.,   6.,   7.,
         7.,   7.,   7.,   7.,   8.,   8.,   8.,   8.,   8.,   8.,   9.,
        10.,  10.,  10.,  11.,  11.,  11.,  12.,  12.,  12.,  13.,  13.,
        14.,  16.,  16.,  16.,  18.,  20.,  21.,  25.,  27.,  27.,  28.,
        35.,  35.,  41.,  43.,  54.,  99., 105.])

In [10]:
Y = Y[:, gene_idx, :]

In [11]:
theta = theta[gene_idx, :]
theta_ = theta.copy()
a0 = theta_[:, 0] # Starting RNA abundance 
a = theta_[:, 1:len(topo.flatten())] 
beta = theta_[:, -2] # Splicing rate
alpha = a * beta[:, None] # These values are divided by splicing rate; removing this factor now
gamma = theta_[:, -1] # Degradation rate

## Reconstruct RNA

In [ ]:
def max_RNAs(Y, gene_idx, frac_added=0.1):
    U_max = np.max(Y[:, gene_idx, 0])
    S_max = np.max(Y[:, gene_idx, 1])
    U_max = (U_max + np.ceil(U_max * frac_added)).astype("int")
    S_max = (S_max + np.ceil(S_max * frac_added)).astype("int")
    return U_max, S_max

### Dense implementation

In [13]:
sparse_cutoff = 50

In [ ]:
X_fwd_per_gene = []
X_bw_per_gene = []
states_per_gene = []

for gene_idx in range(0, Y.shape[1]):
    # Set max # of RNAs based on observed values for a given gene
    U_max, S_max = max_RNAs(Y, gene_idx)

    print("==================================")
    print("Starting gene", gene_idx)
    print("U_max, S_max:", U_max, S_max)
    
    states, index_for = enumerate_states(U_max, S_max)

    beta_j = beta[gene_idx]
    gamma_j = gamma[gene_idx] 
    alpha_j = alpha[gene_idx, :]
    Y_j = Y[:, gene_idx, :]

    # Make generator matrix (per transcription rate)
    A = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha_j[i], beta_j, gamma_j)
        if (U_max > sparse_cutoff) or (S_max > sparse_cutoff):
            A1 = create_transition_matrix_sparse(rxns, states, index_for, U_max, S_max)
        else:
            A1 = create_transition_matrix(rxns, states, index_for, U_max, S_max)
        A.append(A1)
    
    # Calculate forward probability distributions (needed for reverse generator)
    alpha0 = a0[gene_idx] * beta_j
    pi = stationary_from_params(alpha0, beta_j, gamma_j, states)
    start = time.perf_counter() 
    X_fwd = forward_distribution_blocked(A, pi, states, t, tau, state_grid)
    end = time.perf_counter()
    print(f"Elapsed time for forward_distribution_blocked(): {end - start:.6f} seconds")
    
    # Calculate backward probability distributions
    
    start = time.perf_counter()
    if (U_max > sparse_cutoff) or (S_max > sparse_cutoff):
        X_bw = backward_distribution_sparse_to_dense(Y_j, Q, states, index_for, t, tau, state_grid)
        end = time.perf_counter()
        print(f"Elapsed time for backward_distribution_sparse_to_dense(): {end - start:.6f} seconds")
    else:
        X_bw = backward_distribution(Y_j, Q, states, index_for, t, tau, state_grid)
        end = time.perf_counter()
        print(f"Elapsed time for backward_distribution(): {end - start:.6f} seconds")

    X_fwd_per_gene.append(X_fwd)
    X_bw_per_gene.append(X_bw)
    states_per_gene.append(states)
    
    get_expm.cache_clear()
    get_A_rev.cache_clear()
    get_expm_rev.cache_clear()
    get_A_rev_sparse_to_dense.cache_clear()
    get_expm_rev_sparse_to_dense.cache_clear()

----------------
Starting gene 0
U_max, S_max: 5 14
Elapsed time for forward_distribution_blocked(): 0.283000 seconds
Elapsed time for backward_distribution(): 2.115906 seconds
----------------
Starting gene 1
U_max, S_max: 16 9
Elapsed time for forward_distribution_blocked(): 0.175466 seconds
Elapsed time for backward_distribution(): 3.339383 seconds
----------------
Starting gene 2
U_max, S_max: 4 8
Elapsed time for forward_distribution_blocked(): 0.092285 seconds
Elapsed time for backward_distribution(): 0.613835 seconds
----------------
Starting gene 3
U_max, S_max: 7 6
Elapsed time for forward_distribution_blocked(): 0.054795 seconds
Elapsed time for backward_distribution(): 0.768272 seconds
----------------
Starting gene 4
U_max, S_max: 4 18
Elapsed time for forward_distribution_blocked(): 0.119282 seconds
Elapsed time for backward_distribution(): 1.591195 seconds
----------------
Starting gene 5
U_max, S_max: 4 48
Elapsed time for forward_distribution_blocked(): 0.933195 seconds

IndexError: index 51 is out of bounds for axis 1 with size 51

In [ ]:
import pickle

with open("eLNPs_var>1.5_208_genes_X_fwd_per_gene.pkl", "wb") as f:
    pickle.dump(X_fwd_per_gene, f, protocol=pickle.HIGHEST_PROTOCOL)
    
with open("eLNPs_var>1.5_208_genes_X_bw_per_gene.pkl", "wb") as f:
    pickle.dump(X_bw_per_gene, f, protocol=pickle.HIGHEST_PROTOCOL)
    
with open("eLNPs_var>1.5_208_genes_states_per_gene.pkl", "wb") as f:
    pickle.dump(states_per_gene, f, protocol=pickle.HIGHEST_PROTOCOL)

### Sparse implementation

In [ ]:
X_bw_per_gene = []
states_per_gene = []
    
for gene_idx in range(0, Y.shape[1]):
    print("Starting gene", gene_idx)

    # Set max # of RNAs based on observed values for a given gene
    U_max, S_max = max_RNAs(Y, gene_idx)
    
    print("U_max, S_max:", U_max, S_max)

    states, index_for = enumerate_states(U_max, S_max)
    
    beta_j = beta[gene_idx]
    gamma_j = gamma[gene_idx] 
    alpha_j = alpha[gene_idx, :]
    
    # Make generator matrix (per transcription rate)
    A = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha_j[i], beta_j, gamma_j)
        A1 = create_transition_matrix_sparse(rxns, states, index_for, U_max, S_max)
        A.append(A1)
    
    # Calculate forward probability distribution (needed for reverse generator)
    alpha0 = a0[gene_idx] * beta_j
    pi = stationary_from_params(alpha0, beta_j, gamma_j, states)
    
    start = time.perf_counter()
    X_fwd = forward_distribution_blocked(A, pi, states, t, tau, state_grid)
    end = time.perf_counter()
    print(f"Elapsed time for forward_distribution_blocked(): {end - start:.6f} seconds")

    # Calculate backward probability distribution

    start = time.perf_counter()
    X_bw = backward_distribution_sparse_to_dense(Y[:, gene_idx, :], Q, states, index_for, t, tau, state_grid)
    end = time.perf_counter()
    print(f"Elapsed time for backward_distribution_sparse_to_dense(): {end - start:.6f} seconds") 
    print("----------------")
    
    X_bw_per_gene.append(X_bw)
    states_per_gene.append(states)
    
    get_A_rev_sparse.cache_clear()

In [22]:
get_A_rev_sparse.cache_clear()

In [ ]:
# Y_observed, Y, theta, rd, true_t, true_l = simulate_RNA(topo, tau, theta[0, :][None, :], n=20000, random_seed=666)